In [6]:
import numpy as np
import pandas as pd
import random
import time
from rapidfuzz import process, fuzz, distance
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import spoa 
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.cluster import HDBSCAN
import statistics
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import DBSCAN
import numpy as np
from scipy.cluster.hierarchy import linkage, fcluster, to_tree
from scipy.spatial.distance import pdist
import matplotlib.pyplot as plt

# --- 2. DATA GENERATION & UTILS ---
def mutate_sequence(seq, error_rate=0.10):
    if error_rate <= 0: return seq
    seq_list = list(seq)
    new_seq = []
    for base in seq_list:
        if random.random() < error_rate:
            r = random.random()
            if r < 0.5: new_seq.append(random.choice("ACGT")) 
            elif r < 0.75: 
                new_seq.append(base)
                new_seq.append(random.choice("ACGT"))
            else: pass 
        else:
            new_seq.append(base)
    return "".join(new_seq)

def gen_dna(k): return "".join(random.choices("ACGT", k=k))

# (Generating fresh data to ensure variables exist for the loop below)
n_items = 3_000
pool_bc = [gen_dna(8) for _ in range(n_items)]
pool_ins = [gen_dna(random.randint(300,500)) for _ in range(n_items)]
n_repeat_bc = 500
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(random.randint(300, 500)) for _ in range(n_repeat_bc)]
pool_bc += pool_bc[0:n_repeat_bc]
pool_ins += [gen_dna(random.randint(300,500)) for _ in range(n_repeat_bc)]

# Add very similar sequence that will defeat clustering!
pool_bc += [mutate_sequence(pool_bc[i], 0.03) for i in range(n_repeat_bc)]
pool_ins += [mutate_sequence(pool_ins[i], 0.02) for i in range(n_repeat_bc)]


data = []
for i in range(len(pool_bc)):
     n_reads = random.randint(5, 20) 
     for _ in range(n_reads):
         data.append({
             "ID": i,
             "Barcode": mutate_sequence(pool_bc[i], 0.01), 
             "Insert": mutate_sequence(pool_ins[i], 0.01)
         })

df = pd.DataFrame(data)

In [41]:

import time
import numpy as np
import pandas as pd
from scipy.stats import mode
from scipy.cluster.hierarchy import dendrogram
from sklearn.metrics import pairwise_distances
from sklearn.cluster import AgglomerativeClustering
import spoa
from numba import njit, prange
# Assuming rapidfuzz or fuzzywuzzy is available based on context
try:
    from rapidfuzz import process, fuzz
except ImportError:
    from fuzzywuzzy import process, fuzz
# ==========================================
# TIMING UTILITY
# ==========================================
class GlobalTimer:
    def __init__(self):
        self.timings = {}
        self.starts = {}
    def start(self, name):
        if name not in self.starts: # Don't overwrite if nested recursion causes restart
            self.starts[name] = time.perf_counter()
    def stop(self, name):
        if name in self.starts:
            elapsed = time.perf_counter() - self.starts[name]
            self.timings[name] = self.timings.get(name, 0) + elapsed
            del self.starts[name]
    def report(self):
        print("\n" + "="*40)
        print(f"{'OPERATION':<30} | {'TIME (s)':<10}")
        print("-" * 43)
        for name, val in sorted(self.timings.items(), key=lambda x: x[1], reverse=True):
            print(f"{name:<30} | {val:.4f}")
        print("="*40 + "\n")
timer = GlobalTimer()
# ==========================================
# CORE FUNCTIONS
# ==========================================
def remove_invariant_columns(msa_int):
    """
    Identifies and removes columns where values do not change (invariant).
    Returns the reduced matrix and the indices of the kept columns.
    """
    if msa_int.size == 0:
        return msa_int, np.array([])
       
    mins = msa_int.min(axis=0)
    maxs = msa_int.max(axis=0)
   
    # If min == max, the column has only one unique value (invariant)
    is_variable = (mins != maxs)
    kept_indices = np.where(is_variable)[0]
   
    msa_reduced = msa_int[:, kept_indices]
    return msa_reduced, kept_indices
def mutate_integers_fast(seed_int_arr, error_rate, n_copies, mutation_choices):
    """
    Mutates an integer array directly using NumPy.
    Orders of magnitude faster than string manipulation.
    """
    L = len(seed_int_arr)
   
    # 1. Tile the seed (Create N copies)
    copies = np.tile(seed_int_arr, (n_copies, 1))
   
    # 2. Generate Error Mask
    mask = np.random.random((n_copies, L)) < error_rate
   
    # 3. Apply Mutations
    n_mutations = np.sum(mask)
    if n_mutations > 0:
        replacements = np.random.choice(mutation_choices, size=n_mutations)
        copies[mask] = replacements
       
    return copies
def encode_msa(msa_strings, alphabet="ACGTN-"):
    timer.start("encode_msa")
    """
    Vectorized encoding using ASCII byte views.
    Assumes inputs are ASCII (standard for DNA).
    """
    N = len(msa_strings)
    if N == 0:
        timer.stop("encode_msa")
        return np.array([]), len(alphabet)
   
    max_len = max(len(s) for s in msa_strings)
   
    # 1. Pad and convert to single byte-string buffer
    padded_block = "".join([s.ljust(max_len, '-') for s in msa_strings]).encode('ascii')
   
    # 2. View as NumPy int8 array directly
    arr_view = np.frombuffer(padded_block, dtype=np.int8).copy()
    arr_view = arr_view.reshape(N, max_len)
   
    # 3. Fast Translation Table
    lookup = np.zeros(256, dtype=np.int8) + (len(alphabet) - 1)
    for idx, char in enumerate(alphabet):
        lookup[ord(char)] = idx
       
    # 4. Apply translation
    msa_int = lookup[arr_view]
   
    timer.stop("encode_msa")
    return msa_int, len(alphabet)
def get_marginals(msa_int, vocab_size):
    # Create on hot matrix
    # Use float32 for faster matrix multiplication later
    one_hot = np.eye(vocab_size, dtype=np.float32)[msa_int]
    # Calculate frequencies of each base/N/- at each position in MSA
    P_i = one_hot.mean(axis=0)
    return one_hot, P_i
@njit(parallel=True, fastmath=True)
def compute_mi_from_joint(joint_probs_flat, P_i_flat, L, A):
    """
    Computes MI from the pre-calculated joint probability matrix.
    joint_probs_flat: (L*A, L*A) matrix containing P(x,y)
    P_i_flat: (L*A) vector containing P(x)
    """
    mi = np.zeros((L, L), dtype=np.float32)
    epsilon = 1e-12
    # Parallel loop over positions
    for i in prange(L):
        for j in range(i + 1, L): # Triangle only (symmetric)
            mi_val = 0.0
           
            # Loop over alphabet characters
            for a in range(A):
                p_x = P_i_flat[i * A + a]
               
                for b in range(A):
                    p_y = P_i_flat[j * A + b]
                   
                    # Look up joint prob directly from flat matrix
                    row_idx = i * A + a
                    col_idx = j * A + b
                    p_xy = joint_probs_flat[row_idx, col_idx]
                   
                    if p_xy > epsilon:
                        product = p_x * p_y
                        if product > epsilon:
                            mi_val += p_xy * np.log(p_xy / product)
           
            mi[i, j] = mi_val
            mi[j, i] = mi_val
    return mi
# def compute_mi_scores_optimized(one_hot, P_i):
#     """
#     Hybrid BLAS + Numba approach.
#     100x faster than pure looping for large N.
#     """
#     timer.start("compute_mi_hybrid_execution")
   
#     N, L, A = one_hot.shape
   
#     # 1. BLAS Matrix Multiplication (The Speedup)
#     # We use matrix mult to count co-occurrences of all character pairs at all positions simultaneously.
#     # Reshape (N, L, A) -> (N, L*A)
#     flat_view = one_hot.reshape(N, L * A)
   
#     # Compute (L*A, N) @ (N, L*A) -> (L*A, L*A)
#     # This replaces the loop over N with highly optimized BLAS routines
#     joint_counts = np.dot(flat_view.T, flat_view)
   
#     # Normalize to probabilities
#     joint_probs_flat = joint_counts / N
   
#     # 2. Numba Summation
#     # Pass the pre-computed joint matrix to Numba to do the log/sum logic
#     result = compute_mi_from_joint(joint_probs_flat, P_i.flatten(), L, A)
   
#     timer.stop("compute_mi_hybrid_execution")
#     return result




def get_elbow_columns(mi_matrix, exclusion_distance=5, verbose=0):
    timer.start("get_elbow_columns")
    L = mi_matrix.shape[0]
    # Note: When using reduced matrix, exclusion_distance refers to indices in the reduced set
    mask = np.triu(np.ones((L, L), dtype=bool), k=exclusion_distance + 1)
    rows, cols = np.where(mask)
    scores = mi_matrix[rows, cols]
   
    if len(scores) == 0:
        timer.stop("get_elbow_columns")
        return np.array([])
    # 1. Sort Descending
    sorted_indices = np.argsort(scores)[::-1]
    sorted_scores = scores[sorted_indices]
   
    # 2. Determine "Signal End" (Noise Floor Truncation)
    noise_floor = np.percentile(scores, 25)
    valid_mask = sorted_scores > noise_floor
    n_signal_points = np.sum(valid_mask)
    min_points = min(len(sorted_scores), 5)
    cutoff_idx = max(n_signal_points, min_points)
   
    # 3. Truncate for Geometry Calculation
    curve_y = sorted_scores[:cutoff_idx]
    n_points = len(curve_y)
   
    if n_points < 3:
        timer.stop("get_elbow_columns")
        return np.unique(np.concatenate([rows[sorted_indices[:n_points]], cols[sorted_indices[:n_points]]]))
    # 4. Standard Kneedle on Truncated Curve
    x_norm = np.linspace(0, 1, n_points)
    y_norm = (curve_y - curve_y.min()) / (curve_y.max() - curve_y.min() + 1e-9)
   
    line_vec = np.array([1.0, -1.0])
    line_vec = line_vec / np.linalg.norm(line_vec)
    vec_from_start = np.stack([x_norm, y_norm - 1.0], axis=1)
    distances = np.abs(vec_from_start[:, 0] * line_vec[1] - vec_from_start[:, 1] * line_vec[0])
   
    # 5. Select
    elbow_idx = np.argmax(distances)
    if elbow_idx == 0:
        for i in range(1, n_points - 1):
            if curve_y[i] >= (curve_y[0] * 0.95):
                elbow_idx = i
            else:
                break
    n_selected = elbow_idx + 1
    selected_indices = sorted_indices[:n_selected]
    selected_columns = np.unique(np.concatenate([rows[selected_indices], cols[selected_indices]]))
   
    timer.stop("get_elbow_columns")
    return selected_columns
def calculate_msa_mi_and_top_cols(barcodes, inserts, verbose=0, mi_exclusion_distance=3):
   
    timer.start("SPOA_alignment_initial")
    _, msa_barcodes = spoa.poa(barcodes, algorithm=0)
    _, msa_inserts = spoa.poa(inserts, algorithm=0)
    timer.stop("SPOA_alignment_initial")
    msa_strings = [b + "-"*1 + i for b, i in zip(msa_barcodes, msa_inserts)]
    len_msa_inserts = len(msa_inserts[0])
    msa_int, vocab_size = encode_msa(msa_strings)
   
    # --- OPTIMIZATION: Filter Invariant Columns ---
    timer.start("Filter_Invariant")
    msa_reduced, kept_indices = remove_invariant_columns(msa_int)
    timer.stop("Filter_Invariant")
    if msa_reduced.shape[1] < 2:
        return msa_int, np.zeros((0,0)), np.array([]), len_msa_inserts
    one_hot, P_i = get_marginals(msa_reduced, vocab_size)
   
    # Run optimized MI only on reduced set
    mi_matrix_reduced = compute_mi_scores_optimized_v2(one_hot, P_i, timer)
    # Calculate top cols on reduced matrix
    top_cols_reduced = get_elbow_columns(mi_matrix_reduced, mi_exclusion_distance, verbose=verbose)
   
    # Map back to original indices
    if len(top_cols_reduced) > 0:
        top_cols_original = kept_indices[top_cols_reduced]
    else:
        top_cols_original = np.array([])
    return msa_int, mi_matrix_reduced, top_cols_original, len_msa_inserts
def plot_dendrogram(model, **kwargs):
    counts = np.zeros(model.children_.shape[0])
    n_samples = len(model.labels_)
    for i, merge in enumerate(model.children_):
        current_count = 0
        for child_idx in merge:
            if child_idx < n_samples:
                current_count += 1
            else:
                current_count += counts[child_idx - n_samples]
        counts[i] = current_count
    linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)
    dendrogram(linkage_matrix, **kwargs)
def cluster_msa_subset(msa_subset, expected_error_rate, error_rate_multiplier=None, jump_thresh=None):
    assert (error_rate_multiplier is None) + (jump_thresh is None) == 1
   
    timer.start("hamming_distance_matrix")
    dist_matrix_subset = pairwise_distances(msa_subset, metric='hamming')
    timer.stop("hamming_distance_matrix")
    if dist_matrix_subset.shape[0] == 1:
        return dist_matrix_subset, np.array([0])
   
    if error_rate_multiplier is not None:
        d_thresh = expected_error_rate * error_rate_multiplier
    else:
        temp_matrix = dist_matrix_subset.copy()
        np.fill_diagonal(temp_matrix, np.inf)
        min_dist = np.min(temp_matrix)
        d_thresh = min_dist * jump_thresh + expected_error_rate
    timer.start("agglomerative_clustering")
    agg = AgglomerativeClustering(
        metric="precomputed",
        linkage="single",
        distance_threshold = d_thresh,
        n_clusters=None)
   
    labels = agg.fit_predict(dist_matrix_subset)
    timer.stop("agglomerative_clustering")
    return dist_matrix_subset, labels
def cluster_barcode_insert_pairs(barcodes, inserts, mi_exclusion_distance, expected_error_rate,
                                 error_rate_multiplier=None, jump_thresh=None, verbose=0, subset_msa=True):
   
    msa_int, mi_matrix, top_cols, len_msa_inserts = calculate_msa_mi_and_top_cols(barcodes, inserts, verbose, mi_exclusion_distance)
    if subset_msa and len(top_cols) > 0:
        msa_subset = msa_int[:, top_cols]
    else:
        msa_subset = msa_int
    dist_matrix_subset, labels = cluster_msa_subset(msa_subset, expected_error_rate, error_rate_multiplier, jump_thresh)
    return msa_int, mi_matrix, msa_subset, dist_matrix_subset, labels, len_msa_inserts
import numpy as np

def fast_consensus(msa_list, min_threshold=0.5, high_conf_threshold=0.85):
    """
    Rapidly computes a consensus sequence from a list of aligned strings using vectorization.
    
    Logic:
      - Upper Case: Dominant base frequency >= high_conf_threshold
      - Lower Case: Dominant base frequency >= min_threshold BUT < high_conf_threshold
      - 'N':        Dominant base frequency < min_threshold (Ambiguous/No Consensus)
      - Gaps (-):   Skipped entirely in the final output string.
      
    Args:
        msa_list (list): List of aligned strings (must be same length).
        min_threshold (float): Minimum frequency to accept a base (else 'N').
        high_conf_threshold (float): Frequency required for UPPER CASE (else lower case).
    
    Returns:
        str: Consensus string (gaps removed).
    """
    if not msa_list:
        return ""

    # 1. Convert input to 2D integer array (ASCII values)
    # Using 'S1' then viewing as uint8 is the fastest way to get numeric ASCII map
    arr = np.array([list(s) for s in msa_list], dtype='S1')
    arr_int = arr.view(np.uint8)
    n_seqs, length = arr_int.shape
    
    # ASCII constants
    GAP = 45       # '-'
    N_VAL = 78     # 'N'
    TO_LOWER = 32  # Add this to convert A->a, C->c (ASCII offset)

    # 2. Parallel Frequency Counting
    # Instead of looping, we broadcast comparisons.
    # Get unique characters present in the entire alignment to avoid checking all 256 ASCIIs
    unique_chars = np.unique(arr_int)
    
    # Create a 3D broadcast: (num_unique, n_seqs, length) -> sum over n_seqs
    # Result shape: (num_unique, length) = counts of each char per column
    counts = (arr_int == unique_chars[:, None, None]).sum(axis=1)
    
    # 3. Determine Winners
    # argmax gives the index of the character with highest count in each column
    max_indices = counts.argmax(axis=0)
    max_counts = counts.max(axis=0)
    
    # Map indices back to actual ASCII values to get the "Raw Consensus"
    consensus_arr = unique_chars[max_indices]
    
    # Calculate frequencies for every column
    freqs = max_counts / n_seqs

    # 4. Apply Logic (Vectorized)
    
    # Mask for Gaps (We preserve them for now to avoid N-ifying them, then strip later)
    is_gap = (consensus_arr == GAP)
    
    # Rule A: Low Confidence / Ambiguity -> Convert to 'N'
    # If the winner has less than min_threshold support (e.g. < 50%), it's 'N'.
    low_conf_mask = (freqs < min_threshold) & (~is_gap)
    consensus_arr[low_conf_mask] = N_VAL
    
    # Rule B: Mid Confidence -> Convert to Lower Case
    # If min <= freq < high, add 32 to ASCII (Upper -> Lower)
    mid_conf_mask = (freqs >= min_threshold) & (freqs < high_conf_threshold) & (~is_gap)
    
    # Only lowercase valid letters (A-Z are 65-90). Protects against lowercasing 'N' or special chars.
    is_upper_alpha = (consensus_arr >= 65) & (consensus_arr <= 90)
    
    # Apply lower casing offset
    consensus_arr[mid_conf_mask & is_upper_alpha] += TO_LOWER

    # 5. Remove Gaps and Convert to String
    # Filter out gaps (ASCII 45) to return the ungapped consensus sequence
    final_int_seq = consensus_arr[consensus_arr != GAP]
    
    return final_int_seq.tobytes().decode('utf-8')

    
def get_poa_consensus(barcode_series, bias_towards=None):
    timer.start("SPOA_Consensus_Total")
    barcode_list = barcode_series.tolist()
    if bias_towards is not None:
        barcode_list.append(bias_towards)
   
    _, msa = spoa.poa(barcode_list, algorithm=1)
    consensus = fast_consensus(msa)
    timer.stop("SPOA_Consensus_Total")
    return consensus
   
def recursive_outlier_removal(all_df, cluster_df, filtered_dist_matrix, expected_error_rate, percentile_th,
                              verbose, error_rate_multiplier=1.2, n_decoys=50):
   
    if len(cluster_df) < 2:
        return cluster_df
    m = filtered_dist_matrix.astype(float)
    m[np.triu_indices_from(m)] = np.nan
    max_val = np.nanmax(m)
    if max_val <= expected_error_rate*error_rate_multiplier:
        return cluster_df
    m = filtered_dist_matrix.astype(float)
    m[np.triu_indices_from(m)] = np.nan
    median_dist = np.nanmedian(m)
    agg = AgglomerativeClustering(
        metric="precomputed",
        linkage="single",
        distance_threshold=median_dist,
        n_clusters=None
    )
    cluster_labels = agg.fit_predict(filtered_dist_matrix)
    unique, counts = np.unique(cluster_labels, return_counts=True)
    largest_cluster_label = unique[np.argmax(counts)]
   
    largest_cluster_indices = np.where(cluster_labels == largest_cluster_label)[0]
    largest_cluster_df = cluster_df.iloc[largest_cluster_indices]
    largest_subclust_dist_matrix = filtered_dist_matrix[np.ix_(largest_cluster_indices, largest_cluster_indices)]
    mean_distances = np.mean(largest_subclust_dist_matrix, axis=1)
    core_member_index = np.argmin(mean_distances)
    best_core_insert = largest_cluster_df['Insert'].iloc[core_member_index]
    # --- TIMING CRITICAL SECTION ---
    timer.start("Fuzzy_CDIST_Calculations")
    all_ratios = process.cdist([best_core_insert], list(cluster_df['Insert']), scorer=fuzz.ratio, dtype=np.float32)[0]
    timer.stop("Fuzzy_CDIST_Calculations")
    outlier_position = np.argmin(all_ratios)
    outlier_ratio = all_ratios[outlier_position]
    outlier_insert = cluster_df['Insert'].iloc[outlier_position]
    random_inserts = all_df['Insert'].sample(n=n_decoys)
    # --- TIMING CRITICAL SECTION ---
    timer.start("Fuzzy_CDIST_Calculations")
    decoy_ratios = process.cdist([outlier_insert], random_inserts, scorer=fuzz.ratio, dtype=np.float32)[0]
    timer.stop("Fuzzy_CDIST_Calculations")
    threshold_ratio = np.percentile(decoy_ratios, percentile_th)
    if outlier_ratio >= threshold_ratio:
        return cluster_df
    else:
        keep_mask = np.arange(len(filtered_dist_matrix)) != outlier_position
        new_dist_matrix = filtered_dist_matrix[np.ix_(keep_mask, keep_mask)]
        new_cluster_df = cluster_df.iloc[keep_mask]
       
        return recursive_outlier_removal(all_df, new_cluster_df,
                                         new_dist_matrix,
                                         expected_error_rate, percentile_th, verbose, error_rate_multiplier, n_decoys)

        
# def calc_max_row_ratio(mi_matrix_reduced, n_total_cols, top_pct=0.01, bottom_pct=0.50):
#     """
#     Calculates the Ratio (Top Mean / Bottom Mean) for each row.
#     Handles 'implicit zeros' from filtered columns to ensure stats are correct relative to original MSA width.
#     """
#     R = mi_matrix_reduced.shape[1]
#     if R == 0: return 0.0
   
#     # Sort the reduced matrix rows (Ascending)
#     # Only sort the rows we have (which are the variant ones)
#     sorted_rows = np.sort(mi_matrix_reduced, axis=1)
   
#     n_zeros = n_total_cols - R
   
#     k_top = max(1, int(n_total_cols * top_pct))
#     k_bottom = max(1, int(n_total_cols * bottom_pct))
   
#     # --- Top Mean Calculation ---
#     if k_top <= R:
#         # Take top k_top values from the reduced matrix (last k_top columns)
#         top_vals = sorted_rows[:, -k_top:]
#         top_mean = np.mean(top_vals, axis=1)
#     else:
#         # We need more values than we have in the reduced matrix.
#         # This means we take ALL R values, plus (k_top - R) implicit zeros.
#         sum_top = np.sum(sorted_rows, axis=1)
#         top_mean = sum_top / k_top
       
#     # --- Bottom Mean Calculation ---
#     if k_bottom <= n_zeros:
#         # All k_bottom values fall into the implicit zero region
#         bottom_mean = np.zeros(R, dtype=mi_matrix_reduced.dtype)
#     else:
#         # We need (k_bottom - n_zeros) values from the ACTUAL data (starting from smallest)
#         n_needed_from_real = k_bottom - n_zeros
       
#         # Take the smallest n_needed values from sorted_rows
#         bottom_vals = sorted_rows[:, :n_needed_from_real]
#         sum_bottom = np.sum(bottom_vals, axis=1)
       
#         # The mean is Sum / k_bottom (the zeros contribute nothing to sum)
#         bottom_mean = sum_bottom / k_bottom
   
#     damping_factor = 0.01
#     ratios = (top_mean + damping_factor) / (bottom_mean + damping_factor)
   
#     return np.max(ratios)


import numpy as np
from numba import njit, prange
# ==========================================
# OPTIMIZED MI COMPUTATION
# ==========================================
@njit(parallel=True, fastmath=True)
def compute_mi_sparse(joint_probs_flat, P_i_flat, L, A, position_pairs):
    """
    Computes MI only for specified position pairs.
    Much faster when you only need a subset of interactions.
   
    position_pairs: Nx2 array of (i, j) pairs to compute
    """
    n_pairs = position_pairs.shape[0]
    mi_values = np.zeros(n_pairs, dtype=np.float32)
    epsilon = 1e-12
    for pair_idx in prange(n_pairs):
        i = position_pairs[pair_idx, 0]
        j = position_pairs[pair_idx, 1]
       
        mi_val = 0.0
        for a in range(A):
            p_x = P_i_flat[i * A + a]
           
            for b in range(A):
                p_y = P_i_flat[j * A + b]
               
                row_idx = i * A + a
                col_idx = j * A + b
                p_xy = joint_probs_flat[row_idx, col_idx]
               
                if p_xy > epsilon:
                    product = p_x * p_y
                    if product > epsilon:
                        mi_val += p_xy * np.log(p_xy / product)
       
        mi_values[pair_idx] = mi_val
   
    return mi_values
@njit(parallel=True, fastmath=True)
def compute_mi_from_joint_optimized(joint_probs_flat, P_i_flat, L, A):
    """
    Optimized version with better memory access patterns.
    """
    mi = np.zeros((L, L), dtype=np.float32)
    epsilon = 1e-12
    # Precompute log(P_i) to avoid redundant calculations
    log_P_i = np.zeros(L * A, dtype=np.float32)
    for idx in range(L * A):
        if P_i_flat[idx] > epsilon:
            log_P_i[idx] = np.log(P_i_flat[idx])
    for i in prange(L):
        for j in range(i + 1, L):
            mi_val = 0.0
           
            for a in range(A):
                p_x = P_i_flat[i * A + a]
                if p_x <= epsilon:
                    continue
               
                log_p_x = log_P_i[i * A + a]
               
                for b in range(A):
                    p_y = P_i_flat[j * A + b]
                    if p_y <= epsilon:
                        continue
                   
                    row_idx = i * A + a
                    col_idx = j * A + b
                    p_xy = joint_probs_flat[row_idx, col_idx]
                   
                    if p_xy > epsilon:
                        log_p_y = log_P_i[j * A + b]
                        mi_val += p_xy * (np.log(p_xy) - log_p_x - log_p_y)
           
            mi[i, j] = mi_val
            mi[j, i] = mi_val
   
    return mi
def compute_mi_scores_optimized_v2(one_hot, P_i, timer):
    """
    Enhanced version with precomputed logs.
    """
    timer.start("compute_mi_hybrid_execution")
   
    N, L, A = one_hot.shape
   
    # BLAS multiplication (unchanged - already optimal)
    flat_view = one_hot.reshape(N, L * A)
    joint_counts = np.dot(flat_view.T, flat_view)
    joint_probs_flat = joint_counts / N
   
    # Use optimized Numba function
    result = compute_mi_from_joint_optimized(joint_probs_flat, P_i.flatten(), L, A)
   
    timer.stop("compute_mi_hybrid_execution")
    return result
# ==========================================
# VECTORIZED MAX ROW RATIO
# ==========================================
def calc_max_row_ratio_vectorized(mi_matrix_reduced, n_total_cols, top_pct=0.01, bottom_pct=0.50):
    """
    Fully vectorized version - removes all Python loops.
    ~10-20x faster for typical sizes.
    """
    R = mi_matrix_reduced.shape[0]
    if R == 0:
        return 0.0
   
    C = mi_matrix_reduced.shape[1]
    if C == 0:
        return 0.0
   
    # Sort once for all rows
    sorted_rows = np.sort(mi_matrix_reduced, axis=1)
   
    n_zeros = n_total_cols - C
    k_top = max(1, int(n_total_cols * top_pct))
    k_bottom = max(1, int(n_total_cols * bottom_pct))
   
    # --- Vectorized Top Mean ---
    if k_top <= C:
        top_mean = np.mean(sorted_rows[:, -k_top:], axis=1)
    else:
        sum_top = np.sum(sorted_rows, axis=1)
        top_mean = sum_top / k_top
   
    # --- Vectorized Bottom Mean ---
    if k_bottom <= n_zeros:
        bottom_mean = np.zeros(R, dtype=np.float32)
    else:
        n_needed = k_bottom - n_zeros
        sum_bottom = np.sum(sorted_rows[:, :n_needed], axis=1)
        bottom_mean = sum_bottom / k_bottom
   
    # Vectorized ratio calculation
    damping = 0.01
    ratios = (top_mean + damping) / (bottom_mean + damping)
   
    return np.max(ratios)
# ==========================================
# OPTIMIZED INTEGRITY CHECK
# ==========================================
def check_cluster_integrity_mi_optimized(cluster_df, expected_error_rate, timer, verbose=0):
    """
    Enhanced version with:
    - Early termination in simulations
    - Reusable encoding mapping
    - Better simulation batching
    """
    timer.start("MI_Simulations")
   
    import spoa
   
    barcodes = cluster_df['Barcode'].tolist()
    inserts = cluster_df['Insert'].tolist()
    n_members = len(barcodes)
   
    if n_members < 10:
        timer.stop("MI_Simulations")
        return False
   
    # Alignment
    timer.start("SPOA_alignment_integrity_check")
    _, msa_barcodes = spoa.poa(barcodes, algorithm=2)
    _, msa_inserts = spoa.poa(inserts, algorithm=2)
    timer.stop("SPOA_alignment_integrity_check")
   
    msa_strings = [b + "-"*1 + i for b, i in zip(msa_barcodes, msa_inserts)]
   
    # Encoding
    # from your_module import encode_msa, remove_invariant_columns, get_marginals
    msa_int, vocab_size = encode_msa(msa_strings)
   
    alphabet = "ACGTN-"
    char_to_int = {c: i for i, c in enumerate(alphabet)}
    valid_mutation_indices = np.array([char_to_int[c] for c in "ACGT"])
   
    # Calculate Real Score
    timer.start("Filter_Invariant")
    msa_reduced, kept_indices = remove_invariant_columns(msa_int)
    timer.stop("Filter_Invariant")
   
    if msa_reduced.shape[1] < 2:
        timer.stop("MI_Simulations")
        return False
   
    one_hot, P_i = get_marginals(msa_reduced, vocab_size)
   
    # Note: compute_mi_scores_optimized_v2 has its own timer internally
    mi_matrix_reduced = compute_mi_scores_optimized_v2(one_hot, P_i, timer)
    real_score = calc_max_row_ratio_vectorized(mi_matrix_reduced, n_total_cols=msa_int.shape[1])
   
    # Simulation Setup
    seed_idx = np.random.randint(n_members)
    seed_int = msa_int[seed_idx]
   
    # Adaptive simulation with early stopping
    stages = [
        (5, 0.90), # 5 sims, stop if real_score < 90th percentile
        (10, 0.95), # 10 more sims, stop if < 95th percentile
        (85, 0.99) # Final 85 sims for 99th percentile
    ]
   
    all_sim_scores = []
   
    for step_count, percentile in stages:
        batch_scores = []
       
        for _ in range(step_count):
            # from your_module import mutate_integers_fast
            synthetic_ints = mutate_integers_fast(seed_int, expected_error_rate, n_members, valid_mutation_indices)
           
            s_reduced, _ = remove_invariant_columns(synthetic_ints)
           
            if s_reduced.shape[1] < 2:
                score = 0.0
            else:
                s_one_hot = np.eye(vocab_size, dtype=np.float32)[s_reduced]
                s_P_i = s_one_hot.mean(axis=0)
                # Note: This inner MI call also has timer, but we want the total time
                s_mi = compute_mi_scores_optimized_v2(s_one_hot, s_P_i, timer)
                score = calc_max_row_ratio_vectorized(s_mi, n_total_cols=synthetic_ints.shape[1])
           
            batch_scores.append(score)
       
        all_sim_scores.extend(batch_scores)
       
        # Early termination check
        threshold = np.percentile(all_sim_scores, percentile * 100)
        if real_score <= threshold:
            timer.stop("MI_Simulations")
            return False # Not an outlier, can stop early
   
    # Final check
    threshold_99 = np.percentile(all_sim_scores, 99)
   
    if real_score > threshold_99:
        if verbose >= 1:
            print(f"!!! WARNING: Cluster MI Outlier Detected! (real={real_score:.4f}, p99={threshold_99:.4f})")
        timer.stop("MI_Simulations")
        return True
   
    timer.stop("MI_Simulations")
    return False
# ==========================================
# ADDITIONAL OPTIMIZATION: BATCH PROCESSING
# ==========================================
@njit(parallel=True)
def mutate_integers_fast_batch(seed_int_arr, error_rate, n_copies_per_batch, n_batches, mutation_choices):
    """
    Generate multiple batches of mutations at once for better cache utilization.
    """
    L = len(seed_int_arr)
    total_copies = n_copies_per_batch * n_batches
   
    # Preallocate entire output
    all_copies = np.empty((total_copies, L), dtype=seed_int_arr.dtype)
   
    # Fill in parallel
    for batch_idx in prange(n_batches):
        start_idx = batch_idx * n_copies_per_batch
        end_idx = start_idx + n_copies_per_batch
       
        # Tile seed
        for i in range(n_copies_per_batch):
            all_copies[start_idx + i] = seed_int_arr
       
        # Generate mutations for this batch
        batch_size = n_copies_per_batch * L
        mask = np.random.random(batch_size) < error_rate
       
        if np.sum(mask) > 0:
            replacements = np.random.choice(mutation_choices, size=np.sum(mask))
           
            # Apply mutations
            flat_view = all_copies[start_idx:end_idx].ravel()
            flat_view[mask] = replacements
   
    return all_copies
   
def full_analysis(sub_df, all_df, barcode_target, percentile_th, expected_error_rate,
                  verbose, mi_exclusion_distance, cluster_again_total_mean_thresh=2,
                  cluster_again_single_mean_thresh=3):
    if len(sub_df) < 2:
        if 'cluster' not in sub_df.columns: sub_df = sub_df.copy(); sub_df['cluster'] = -1
        return sub_df
    barcodes = sub_df['Barcode'].tolist()
    inserts = sub_df['Insert'].tolist()
   
    msa_int, mi_matrix, msa_subset, dist_matrix_subset, labels, len_msa_inserts = cluster_barcode_insert_pairs(
                                                                            barcodes,
                                                                            inserts,
                                                                            mi_exclusion_distance,
                                                                            expected_error_rate,
                                                                            error_rate_multiplier=3,
                                                                            verbose=0,
                                                                            subset_msa=False)
    labels = labels + hash(frozenset(sub_df.index)) % 10_000_000
    sub_df['cluster'] = labels
    cluster_results = []
    for label in list(set(labels)):
        this_cluster = sub_df[sub_df['cluster'] == label]
        indices = np.where(labels == label)[0]
        filtered_dist_matrix = dist_matrix_subset[np.ix_(indices, indices)]
        final_processed_cluster = recursive_outlier_removal(all_df, this_cluster, filtered_dist_matrix,
                                                            expected_error_rate, percentile_th, verbose)
        # print("Checking MI")
        if len(final_processed_cluster) >= 8 and label != -1:
            is_suspicious = check_cluster_integrity_mi_hyper_optimized(final_processed_cluster, expected_error_rate, timer, verbose=verbose)
           
            final_processed_cluster = final_processed_cluster.copy()
            final_processed_cluster['mi_warning'] = is_suspicious
           
            if is_suspicious:
                print(f"Cluster {label} may be formed of two highly similar sequences")
        else:
            final_processed_cluster = final_processed_cluster.copy()
            final_processed_cluster['mi_warning'] = False
               
        cluster_results.append(final_processed_cluster)
    result = pd.concat(cluster_results)
   
    # === PRINT TIMING REPORT ===
    timer.report()
   
    return result
    
import numpy as np
from rapidfuzz import fuzz

import numpy as np
from rapidfuzz import fuzz

def filter_result_df(result, barcode_target, verbose, 
                     filter_target=True, 
                     min_barcode_ratio_hq=0.975):
    """
    Args:
        min_barcode_ratio_hq (float): Threshold (0.0-1.0). If the mean fuzz ratio of barcodes 
                                      in a cluster >= this value, 'barcodes_highly_consistent' is True.
    """
    result = result.copy()
    
    # 1. Optimized Consensus Generation
    unique_clusters = result['cluster'].unique()
    consensus_map = {}
    
    # Use timer if it exists in scope, otherwise ignore
    try: timer.start("filter_consensus_generation")
    except: pass
    
    for cl in unique_clusters:
        subset = result.loc[result['cluster'] == cl, 'Barcode']
        # Ensure get_poa_consensus handles the subset correctly
        consensus_map[cl] = get_poa_consensus(subset)
        
    try: timer.stop("filter_consensus_generation")
    except: pass
    
    result['cluster_consensus_bc'] = result['cluster'].map(consensus_map)
    
    # 2. Filter by Target Barcode (Optional - keeps rows, drops others)
    if filter_target:
        result = result[
            (result['cluster_consensus_bc'] == barcode_target) |
            ( (result.groupby('cluster')['Barcode'].transform('size') == 2) & 
              (result.groupby('cluster')['Barcode'].transform(lambda x: (x == barcode_target).any()))
            )
        ]
    
    if result.empty:
        return result

    # 3. Calculate Consistency and Add Flag (New)
    # Define a helper to calculate the mean score for a single cluster
    def get_mean_consistency(df_chunk):
        consensus = df_chunk['cluster_consensus_bc'].iloc[0]
        # Handle edge case where consensus might be empty
        if not consensus: 
            return 0.0
        
        # Calculate mean ratio of all barcodes in this cluster to the consensus
        # fuzz.ratio returns 0-100, we normalize to 0.0-1.0
        scores = [fuzz.ratio(b, consensus) for b in df_chunk['Barcode']]
        return np.mean(scores) / 100.0

    # Apply to each cluster to get a Series: index=cluster_id, value=mean_score
    cluster_scores = result.groupby('cluster').apply(get_mean_consistency)
    
    # Map the scores back to the main dataframe
    result['cluster_mean_barcode_sim'] = result['cluster'].map(cluster_scores)
    
    # Create the boolean flag column
    result['barcodes_consistent'] = result['cluster_mean_barcode_sim'] >= min_barcode_ratio_hq

    # 4. Final Cleanup
    # Generate Insert Consensus
    result['cluster_consensus_insert'] = result.groupby('cluster')['Insert'].transform(get_poa_consensus)
    
    # Renumber clusters sequentially (0, 1, 2...)
    unique = [u for u in sorted(result['cluster'].unique()) if u != -1]
    mapping = {old: new for new, old in enumerate(unique)}
    result['cluster'] = result['cluster'].map(lambda x: mapping.get(x, -1))
    
    return result
    
from numba import njit, prange
import numpy as np


import numpy as np
from numba import njit, prange

# ==========================================
# 1. PRECOMPUTE MI LOOKUP TABLE
# ==========================================
@njit
def create_mi_lookup(n_members):
    """
    Creates a 3D lookup table for MI terms.
    Dimensions: [count_xy, count_x, count_y]
    Pre-calculates: (N_xy/N) * log((N_xy*N) / (N_x*N_y))
    Access is O(1) integer lookup.
    """
    # Max possible count is n_members.
    # We add +1 for 0-based indexing.
    lut = np.zeros((n_members + 1, n_members + 1, n_members + 1), dtype=np.float32)
    
    N = float(n_members)
    epsilon = 1e-12
    
    for c_xy in range(1, n_members + 1):
        for c_x in range(1, n_members + 1):
            for c_y in range(1, n_members + 1):
                # Valid counts only (count_xy cannot exceed x or y)
                if c_xy <= c_x and c_xy <= c_y:
                    p_xy = c_xy / N
                    p_x = c_x / N
                    p_y = c_y / N
                    
                    term = p_xy * np.log(p_xy / (p_x * p_y))
                    lut[c_xy, c_x, c_y] = term
                    
    return lut

# ==========================================
# 2. THE CORE KERNEL (Integer Only)
# ==========================================
@njit(fastmath=True)
def _fast_score_kernel(msa, n_members, L, variant_indices, n_var, lut, top_pct, bottom_pct):
    """
    Computes Max Row Ratio using:
    - Integer counting (No OneHot, No Dot Product)
    - Lookup Table (No Log, No Div)
    """
    # 1. Compute Joint Counts & Marginals
    # We use a flat buffer for the counts. 
    # Size: n_var * 5 (ACGTN)
    # We only care about columns mapped in variant_indices
    
    # vocab size fixed to 5 (ACGTN) for speed
    V = 5 
    
    # Counts: (n_var, V)
    marginals = np.zeros((n_var, V), dtype=np.int32)
    
    # Joint: (n_var, n_var, V, V)
    # For small n_var (<50), this fits in cache.
    # We only fill upper triangle.
    joint = np.zeros((n_var, n_var, V, V), dtype=np.int32)
    
    for r in range(n_members):
        # We iterate variants, not full L
        for i in range(n_var):
            col_i = variant_indices[i]
            char_i = msa[r, col_i]
            if char_i >= V: continue # Skip gaps/invalid if any
            
            marginals[i, char_i] += 1
            
            # Cross terms
            for j in range(i + 1, n_var):
                col_j = variant_indices[j]
                char_j = msa[r, col_j]
                if char_j >= V: continue
                
                joint[i, j, char_i, char_j] += 1

    # 2. Compute Score using LUT
    max_ratio = 0.0
    damping = 0.01
    
    # Limits
    k_top = int(L * top_pct)
    if k_top < 1: k_top = 1
    k_bottom = int(L * bottom_pct)
    if k_bottom < 1: k_bottom = 1
    n_zeros_implicit = L - n_var
    
    # Reusable row buffer
    row_mis = np.zeros(n_var, dtype=np.float32)
    
    for i in range(n_var):
        # Calculate MI row i vs all j
        for j in range(n_var):
            if i == j: 
                row_mis[j] = 0.0
                continue
            
            # Sort indices for Upper Triangle Access
            if i < j:
                u, v = i, j
            else:
                u, v = j, i
                
            mi_val = 0.0
            
            # Sum 5x5 alphabet
            for a in range(V):
                c_x = marginals[i, a]
                if c_x == 0: continue
                
                for b in range(V):
                    c_y = marginals[j, b]
                    if c_y == 0: continue
                    
                    # Access Joint (always u, v)
                    # Mapping: if i<j (u=i, v=j), joint[u,v,a,b] is count(x=a, y=b)
                    # if i>j (u=j, v=i), joint[u,v,b,a] is count(x=b, y=a) -> Same thing symmetric
                    if i < j:
                        c_xy = joint[u, v, a, b]
                    else:
                        c_xy = joint[u, v, b, a]
                        
                    if c_xy > 0:
                        # O(1) Lookup
                        mi_val += lut[c_xy, c_x, c_y]
            
            row_mis[j] = mi_val
            
        # 3. Ratio Logic (Inline Sort)
        row_mis = np.sort(row_mis)
        
        # Top Mean
        take_cnt = min(k_top, n_var)
        sum_top = 0.0
        if take_cnt > 0:
            for k in range(take_cnt):
                sum_top += row_mis[n_var - 1 - k]
        top_mean = sum_top / k_top
        
        # Bottom Mean
        sum_bot = 0.0
        if k_bottom > n_zeros_implicit:
            needed = k_bottom - n_zeros_implicit
            take_bot = min(needed, n_var)
            for k in range(take_bot):
                sum_bot += row_mis[k]
            bottom_mean = sum_bot / k_bottom
        else:
            bottom_mean = 0.0
            
        ratio = (top_mean + damping) / (bottom_mean + damping)
        if ratio > max_ratio:
            max_ratio = ratio
            
    return max_ratio

# ==========================================
# 3. PARALLEL SIMULATION DRIVER
# ==========================================
@njit(parallel=True, fastmath=True)
def run_parallel_sims(seed_int, n_sims, n_members, error_rate, valid_muts, lut, top_pct, bot_pct):
    scores = np.zeros(n_sims, dtype=np.float32)
    L = len(seed_int)
    
    # Parallel Loop: Each thread gets its own simulation
    for s in prange(n_sims):
        # 1. Local Allocation (Cheap in Numba)
        msa = np.empty((n_members, L), dtype=np.int8)
        var_inds = np.empty(L, dtype=np.int32)
        
        # 2. Mutate (Manual Loop for speed)
        # We fill msa and track variants in one go if possible, 
        # but 2-pass is cleaner and safer.
        
        # Copy Seed
        for r in range(n_members):
            msa[r, :] = seed_int
            
        # Apply Mutations
        # We iterate cells.
        for r in range(n_members):
            for c in range(L):
                # Fast RNG
                if np.random.random() < error_rate:
                    # random choice 0-3
                    ri = np.random.randint(0, 4)
                    msa[r, c] = valid_muts[ri]
        
        # 3. Find Variants
        n_var = 0
        for c in range(L):
            v0 = msa[0, c]
            is_var = False
            for r in range(1, n_members):
                if msa[r, c] != v0:
                    is_var = True
                    break
            if is_var:
                var_inds[n_var] = c
                n_var += 1
                
        if n_var < 2:
            scores[s] = 0.0
        else:
            scores[s] = _fast_score_kernel(msa, n_members, L, var_inds, n_var, lut, top_pct, bot_pct)
            
    return scores


import spoa
import random
import numpy as np
import spoa
import random

def fast_spoa(sequences, algorithm=1, mode='msa', max_initial_reads=50):
    """
    Corrected wrapper for the functional spoa.poa() binding.
    """
    # Ensure input is a standard list of strings
    if not isinstance(sequences, list):
        sequences = list(sequences)
        
    if not sequences:
        return "", []

    # --- PATH 1: Consensus Only (Fast / Approximate) ---
    # Use this for 'SPOA_alignment_initial' to break the 10s bottleneck.
    # We subsample to keep it fast.
    if mode == 'consensus' and len(sequences) > max_initial_reads:
        subset = random.sample(sequences, max_initial_reads)
        try:
            # Run SPOA on just the small subset
            consensus, _ = spoa.poa(subset, algorithm=algorithm)
            return consensus, [] # Return empty MSA because subset MSA won't match full input
        except Exception as e:
            print(f"SPOA Subset Error: {e}")
            return "", []

    # --- PATH 2: Full MSA (Required for MI Check) ---
    # Use this for 'check_cluster_integrity'. We CANNOT subsample here 
    # because we need the MSA to match the input rows 1-to-1.
    try:
        # The library calculates everything in one go
        consensus, msa = spoa.poa(sequences, algorithm=algorithm)
        return consensus, msa
    except Exception as e:
        print(f"SPOA Full Error: {e}")
        return "", []

        
# ==========================================
# 4. MAIN FUNCTION
# ==========================================
def check_cluster_integrity_mi_hyper_optimized(cluster_df, expected_error_rate, timer, verbose=0):
    timer.start("MI_Integrity_Check_Total")
    
    barcodes = cluster_df['Barcode'].tolist()
    inserts = cluster_df['Insert'].tolist()
    n_members = len(barcodes)
    
    if n_members < 10:
        timer.stop("MI_Integrity_Check_Total")
        return False
    
    # --- Alignment (Standard) ---
    timer.start("SPOA_alignment")
    _, msa_barcodes = spoa.poa(barcodes, algorithm=1)
    _, msa_inserts = spoa.poa(inserts, algorithm=1)
    timer.stop("SPOA_alignment")
    
    msa_strings = [b + "-"*1 + i for b, i in zip(msa_barcodes, msa_inserts)]
    msa_int, vocab_size = encode_msa(msa_strings)
    
    # Prep constants
    alphabet = "ACGTN-"
    char_to_int = {c: i for i, c in enumerate(alphabet)}
    valid_muts = np.array([char_to_int[c] for c in "ACGT"], dtype=np.int8)
    
    # --- PRECOMPUTE LUT (Once per cluster) ---
    # This is incredibly fast (micro-seconds) for N=50
    lut = create_mi_lookup(n_members)

    # --- REAL SCORE ---
    # We use the same fast kernel! 
    timer.start("Real_Score")
    # Identify variants in real MSA
    L = msa_int.shape[1]
    var_inds = np.empty(L, dtype=np.int32)
    n_var = 0
    # Manual variant finding for real MSA to match kernel format
    # (Or use your existing remove_invariant_columns but return full indices)
    # Let's just do it quickly here:
    msa_contig = np.ascontiguousarray(msa_int)
    
    # Simple python loop is fine for 1 call, or numba it. 
    # Let's assume we pass it to the kernel, but we need n_var.
    # We'll use a small helper or just the existing tool:
    msa_reduced, kept_indices = remove_invariant_columns(msa_int)
    n_var_real = len(kept_indices)
    
    if n_var_real < 2:
        real_score = 0.0
    else:
        # We need to pass the FULL MSA and the indices to the kernel
        # actually, the kernel takes the full MSA and indices.
        # But for the real one, we already have reduced.
        # We can construct a dummy "full" MSA or just adapt.
        # EASIER: Just use the same parallel kernel with n_sims=1 and no mutation? 
        # No, let's just use the kernel logic directly on the reduced MSA.
        # Note: The kernel expects full MSA and indices.
        # We can just pass msa_int and the kept_indices.
        var_inds_real = kept_indices.astype(np.int32)
        real_score = _fast_score_kernel(msa_contig, n_members, L, var_inds_real, n_var_real, lut, 0.05, 0.50)
    timer.stop("Real_Score")
    
    # --- SIMULATIONS ---
    timer.start("Sim_Parallel")
    seed_idx = np.random.randint(n_members)
    seed_int = msa_contig[seed_idx]
    
    sim_scores = run_parallel_sims(
        seed_int, 
        100, 
        n_members, 
        expected_error_rate, 
        valid_muts, 
        lut, 
        0.05, 
        0.50
    )
    timer.stop("Sim_Parallel")
    
    # --- CHECK ---
    threshold_99 = np.percentile(sim_scores, 99)
    is_outlier = real_score > threshold_99
    
    if is_outlier and verbose >= 1:
        print(f"!!! Outlier: Real={real_score:.4f} > P99={threshold_99:.4f}")
        
    timer.stop("MI_Integrity_Check_Total")
    return is_outlier

import numpy as np
from numba import njit, prange

# ==========================================
# 1. LOOKUP TABLE (Unchanged)
# ==========================================
@njit
def create_mi_lookup(n_members):
    lut = np.zeros((n_members + 1, n_members + 1, n_members + 1), dtype=np.float32)
    N = float(n_members)
    epsilon = 1e-12
    
    for c_xy in range(1, n_members + 1):
        for c_x in range(1, n_members + 1):
            for c_y in range(1, n_members + 1):
                # Valid counts only
                if c_xy <= c_x and c_xy <= c_y:
                    p_xy = c_xy / N
                    p_x = c_x / N
                    p_y = c_y / N
                    
                    term = p_xy * np.log(p_xy / (p_x * p_y))
                    lut[c_xy, c_x, c_y] = term
    return lut

# ==========================================
# 2. ROBUST POPCOUNT
# ==========================================
@njit(inline='always')
def popcount64(x):
    """
    SWAR algorithm for 64-bit population count.
    Inline always to ensure it compiles to optimal assembly.
    """
    x = np.uint64(x)
    m1 = np.uint64(0x5555555555555555)
    m2 = np.uint64(0x3333333333333333)
    m4 = np.uint64(0x0f0f0f0f0f0f0f0f)
    
    x -= (x >> np.uint64(1)) & m1
    x = (x & m2) + ((x >> np.uint64(2)) & m2)
    x = (x + (x >> np.uint64(4))) & m4
    x = (x * np.uint64(0x0101010101010101)) >> np.uint64(56)
    return int(x)

# ==========================================
# 3. BITWISE SCORE KERNEL (Memory Safe)
# ==========================================
@njit(fastmath=True)
def _score_bitwise_internal(bitmaps, n_chunks, L, n_var, lut, top_pct, bottom_pct):
    """
    Computes MI from pre-filled bitmaps.
    bitmaps shape: (4, n_var, n_chunks) OR (4, L, n_chunks)
    We only iterate up to n_var.
    """
    # 1. Marginals (Count 1s per column per base)
    # Stack-allocated small array
    marginals = np.zeros((n_var, 4), dtype=np.int32)
    
    for i in range(n_var):
        for b in range(4):
            count = 0
            for k in range(n_chunks):
                val = bitmaps[b, i, k]
                if val > 0:
                    count += popcount64(val)
            marginals[i, b] = count

    # 2. Pairwise MI
    row_mis = np.zeros(n_var, dtype=np.float32)
    max_ratio = 0.0
    damping = 0.01
    
    k_top = max(1, int(L * top_pct))
    k_bottom = max(1, int(L * bottom_pct))
    n_zeros_implicit = L - n_var
    
    for i in range(n_var):
        # We calculate row i against all j
        for j in range(n_var):
            if i == j: 
                row_mis[j] = 0.0
                continue
                
            mi_val = 0.0
            
            # 4x4 Base Loop
            for b1 in range(4):
                c_x = marginals[i, b1]
                if c_x == 0: continue
                
                for b2 in range(4):
                    c_y = marginals[j, b2]
                    if c_y == 0: continue
                    
                    # Bitwise Intersection
                    c_xy = 0
                    for k in range(n_chunks):
                        bits = bitmaps[b1, i, k] & bitmaps[b2, j, k]
                        if bits > 0:
                            c_xy += popcount64(bits)
                            
                    if c_xy > 0:
                        mi_val += lut[c_xy, c_x, c_y]
                        
            row_mis[j] = mi_val
            
        # 3. Ratio Logic (Inline)
        # Note: sorting a small array (20-50 floats) is very fast
        row_mis = np.sort(row_mis)
        
        # Top Mean
        take_top = min(k_top, n_var)
        sum_top = 0.0
        if take_top > 0:
            for k in range(take_top):
                sum_top += row_mis[n_var - 1 - k]
        top_mean = sum_top / k_top
        
        # Bottom Mean
        sum_bot = 0.0
        if k_bottom > n_zeros_implicit:
            needed = k_bottom - n_zeros_implicit
            take_bot = min(needed, n_var)
            for k in range(take_bot):
                sum_bot += row_mis[k]
            bottom_mean = sum_bot / k_bottom
        else:
            bottom_mean = 0.0
            
        ratio = (top_mean + damping) / (bottom_mean + damping)
        if ratio > max_ratio:
            max_ratio = ratio
            
    return max_ratio

# ==========================================
# 4. PARALLEL SIMULATIONS (Crash Proof)
# ==========================================
@njit(parallel=True, fastmath=True)
def run_bitwise_sims_robust(seed_int, n_sims, n_members, error_rate, valid_muts, lut, top_pct, bot_pct):
    scores = np.zeros(n_sims, dtype=np.float32)
    L = len(seed_int)
    
    # Calculate chunks once
    n_chunks = (n_members + 63) // 64
    
    # Parallel Loop
    for s in prange(n_sims):
        # ALLOCATIONS:
        # We allocate MAX size buffers (based on L) to avoid dynamic resizing.
        # This stability prevents the allocator crash.
        
        # MSA Buffer: (N, L)
        msa = np.empty((n_members, L), dtype=np.int8)
        
        # Variant Index Buffer: (L)
        var_inds = np.empty(L, dtype=np.int32)
        
        # Bitmap Buffer: (4, L, n_chunks) - Fixed size!
        # Initialized to zero
        bitmaps = np.zeros((4, L, n_chunks), dtype=np.uint64)
        
        # 1. Reset & Mutate
        # Copy Seed
        for r in range(n_members):
            msa[r, :] = seed_int
            
        # Apply Random Mutations
        for r in range(n_members):
            for c in range(L):
                if np.random.random() < error_rate:
                    ri = np.random.randint(0, 4)
                    msa[r, c] = valid_muts[ri]
        
        # 2. Identify Variants & Pack Bits Inline
        # We combine these steps to maximize cache usage
        n_var = 0
        
        for c in range(L):
            # Check variance
            v0 = msa[0, c]
            is_var = False
            for r in range(1, n_members):
                if msa[r, c] != v0:
                    is_var = True
                    break
            
            if is_var:
                # Store this column index in our map (for logic consistency)
                var_inds[n_var] = c
                
                # Pack this column into bitmaps immediately
                # n_var is the index in the bitmap array
                for r in range(n_members):
                    char = msa[r, c]
                    if char > 3: continue # Skip gaps/N
                    
                    chunk = r // 64
                    bit = r % 64
                    
                    # Set bit
                    bitmaps[char, n_var, chunk] |= (np.uint64(1) << np.uint64(bit))
                
                n_var += 1
        
        # 3. Score
        if n_var < 2:
            scores[s] = 0.0
        else:
            scores[s] = _score_bitwise_internal(bitmaps, n_chunks, L, n_var, lut, top_pct, bot_pct)
            
    return scores

# ==========================================
# 5. INTEGRATION
# ==========================================
def check_cluster_integrity_mi_hyper_optimized(cluster_df, expected_error_rate, timer, verbose=0):
    timer.start("MI_Integrity_Check_Total")
    
    barcodes = cluster_df['Barcode'].tolist()
    inserts = cluster_df['Insert'].tolist()
    n_members = len(barcodes)
    
    if n_members < 10:
        timer.stop("MI_Integrity_Check_Total")
        return False
    
    # 1. Alignment
    timer.start("SPOA_alignment")
    _, msa_barcodes = spoa.poa(barcodes, algorithm=1)
    _, msa_inserts = fast_spoa(inserts, algorithm=1)
    timer.stop("SPOA_alignment")
    
    msa_strings = [b + "-"*1 + i for b, i in zip(msa_barcodes, msa_inserts)]
    msa_int, vocab_size = encode_msa(msa_strings)
    
    alphabet = "ACGTN-"
    char_to_int = {c: i for i, c in enumerate(alphabet)}
    valid_muts = np.array([char_to_int[c] for c in "ACGT"], dtype=np.int8)
    
    # 2. LUT
    lut = create_mi_lookup(n_members)

    # 3. Real Score
    timer.start("Real_Score")
    msa_reduced, kept_indices = remove_invariant_columns(msa_int)
    n_var_real = len(kept_indices)
    L_real = msa_int.shape[1]
    
    if n_var_real < 2:
        real_score = 0.0
    else:
        # Pack real data
        n_chunks = (n_members + 63) // 64
        # Allocate fixed L size or just n_var_real size (single thread is safe)
        bitmaps = np.zeros((4, n_var_real, n_chunks), dtype=np.uint64)
        msa_contig = np.ascontiguousarray(msa_reduced)
        
        # Manual pack for real data
        for i in range(n_var_real):
            # msa_reduced already contains only variant columns
            # so we iterate 0..n_var_real directly
            for r in range(n_members):
                char = msa_contig[r, i]
                if char > 3: continue
                chunk = r // 64
                bit = r % 64
                bitmaps[char, i, chunk] |= (np.uint64(1) << np.uint64(bit))
                
        real_score = _score_bitwise_internal(bitmaps, n_chunks, L_real, n_var_real, lut, 0.01, 0.50)
    timer.stop("Real_Score")
    
    # 4. Simulations
    timer.start("Sim_Parallel")
    seed_idx = np.random.randint(n_members)
    seed_int = np.ascontiguousarray(msa_int[seed_idx])
    
    sim_scores = run_bitwise_sims_robust(
        seed_int, 
        100, 
        n_members, 
        expected_error_rate, 
        valid_muts, 
        lut, 
        0.05, 
        0.50
    )
    timer.stop("Sim_Parallel")
    
    threshold_99 = np.percentile(sim_scores, 99)
    is_outlier = real_score > threshold_99
    
    if is_outlier and verbose >= 1:
        print(f"!!! Outlier: Real={real_score:.4f} > P99={threshold_99:.4f}")
        
    timer.stop("MI_Integrity_Check_Total")
    return is_outlier



#  ===== SPEE



In [37]:
# ==== PARAMS =====
from sklearn.manifold import MDS
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import MDS
import numpy as np
from sklearn.cluster import OPTICS
from sklearn.cluster import SpectralClustering
from scipy.linalg import eigh
from scipy.cluster.hierarchy import linkage, fcluster
from sklearn.cluster import AffinityPropagation
import numpy as np
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import squareform
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import dendrogram
from scipy.optimize import curve_fit

verbose = 1
percentile_th = 95
expected_error_rate = 0.03
PLOTS = False
mi_exclusion_distance=3
n_sims = 150
sim_percentile = 0.1

# ==== RUN =====

barcode_counts = df['Barcode'].value_counts()
all_barcodes = np.array(barcode_counts.index.tolist())     
    

for i, (top_bc, count) in enumerate(barcode_counts.head(20).items()):

    if verbose >= 1:
        print(f"\n\n=============\n{top_bc}")
        print(f"Processing Top Barcode #{i+1}: {top_bc} (Count: {count})")

    # top_bc = "TATCATCC" # Force specific barcode if needed for debug
    
    # RapidFuzz
    scores = process.cdist([top_bc], all_barcodes, scorer=fuzz.ratio, dtype=np.float32)[0]
    candidate_bcs = all_barcodes[np.where(scores > 93)[0]]
    filtered_df = df[df['Barcode'].isin(candidate_bcs)].copy()

    print(f"length of filtered df: {len(filtered_df)}")

    # filtered_df = pd.read_csv("~/Downloads/fml.csv")

    if filtered_df.empty: continue

    if verbose >= 2:
        print(f"  > RapidFuzz gathered {len(filtered_df)} reads")

    result_df = full_analysis(filtered_df, df, top_bc, 
                                          percentile_th=percentile_th, verbose=verbose, expected_error_rate=expected_error_rate,
                             mi_exclusion_distance=mi_exclusion_distance)

    print(result_df)

    result_df2 = filter_result_df(result_df, top_bc, verbose)

    if verbose >= 1:
        # Create the count DataFrame first (using the recommended groupby method)
        combo_counts_df = result_df2.groupby(['ID', 'cluster', 'mi_warning', 'barcodes_consistent']).size().reset_index(name='count')
        
        # Now, sort the DataFrame
        combo_counts_df.sort_values(by=['cluster', 'count'], ascending=[True, False], inplace=True)
        
        # Print the final result
        print(combo_counts_df)

result_df2



TGGGGTCC
Processing Top Barcode #1: TGGGGTCC (Count: 107)
length of filtered df: 109
!!! Outlier: Real=26.8879 > P99=16.0239
Cluster 440006 may be formed of two highly similar sequences

OPERATION                      | TIME (s)  
-------------------------------------------
MI_Integrity_Check_Total       | 2.2860
Sim_Parallel                   | 1.7278
compute_mi_hybrid_execution    | 0.6304
Real_Score                     | 0.4000
SPOA_alignment_initial         | 0.1887
SPOA_alignment                 | 0.0714
get_elbow_columns              | 0.0299
hamming_distance_matrix        | 0.0034
encode_msa                     | 0.0013
agglomerative_clustering       | 0.0007
Filter_Invariant               | 0.0001

         ID   Barcode                                             Insert  \
46137  3713  TGGGGTCC  GCTCGGATCTTTCAATCTCGGCCACGTCCTGCAAGTGGCCTTGCTA...   
46138  3713  TGGGGTCC  GCTCGGATCTTTCAATCTCGGCCACGTCCTGCAAGTGGCCTTGCTA...   
46139  3713  TGGGGTCC  GCTCGGATCTTTCAATCTCGGCCACGTCCTG

/var/folders/kr/crxb3xqd4wv1ysx_0t4zgxv80000gn/T/ipykernel_10028/3654542128.py:837: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cluster_scores = result.groupby('cluster').apply(get_mean_consistency)


!!! Outlier: Real=63.3655 > P99=25.5963
Cluster 7640098 may be formed of two highly similar sequences

OPERATION                      | TIME (s)  
-------------------------------------------
MI_Integrity_Check_Total       | 2.4353
Sim_Parallel                   | 1.8048
compute_mi_hybrid_execution    | 0.6991
Real_Score                     | 0.4049
SPOA_alignment_initial         | 0.3353
SPOA_alignment                 | 0.1366
SPOA_Consensus_Total           | 0.1040
get_elbow_columns              | 0.0558
hamming_distance_matrix        | 0.0061
encode_msa                     | 0.0031
filter_consensus_generation    | 0.0017
agglomerative_clustering       | 0.0012
Fuzzy_CDIST_Calculations       | 0.0003
Filter_Invariant               | 0.0003

         ID   Barcode                                             Insert  \
46565  3747  ATTGATCG  AGACGACCTCAAAGCTAACTACGACAAGAGAGCTGCCCTATTAGGG...   
46566  3747  ATTGATCG  AGACGACCTCAAAGCTAACTACGACAAGAGAGCTGCCCTATTAGGG...   
46567  3747  ATTGATC

/var/folders/kr/crxb3xqd4wv1ysx_0t4zgxv80000gn/T/ipykernel_10028/3654542128.py:837: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cluster_scores = result.groupby('cluster').apply(get_mean_consistency)


!!! Outlier: Real=31.2179 > P99=20.8536
Cluster 7258793 may be formed of two highly similar sequences
!!! Outlier: Real=61.1366 > P99=20.0639
Cluster 7258794 may be formed of two highly similar sequences

OPERATION                      | TIME (s)  
-------------------------------------------
MI_Integrity_Check_Total       | 2.6227
Sim_Parallel                   | 1.9060
compute_mi_hybrid_execution    | 0.7884
SPOA_alignment_initial         | 0.5357
Real_Score                     | 0.4113
SPOA_alignment                 | 0.2149
SPOA_Consensus_Total           | 0.1943
get_elbow_columns              | 0.0921
hamming_distance_matrix        | 0.0090
encode_msa                     | 0.0042
filter_consensus_generation    | 0.0035
agglomerative_clustering       | 0.0017
Fuzzy_CDIST_Calculations       | 0.0006
Filter_Invariant               | 0.0005

         ID   Barcode                                             Insert  \
40462  3257  AATAAGTT  AGTGTGTAGGGGATCAACAGCCAGGGACAAACGGGACACTCTGTCT.

...

!!! Outlier: Real=57.1072 > P99=16.0820
Cluster 7924229 may be formed of two highly similar sequences

OPERATION                      | TIME (s)  
-------------------------------------------
MI_Integrity_Check_Total       | 4.9259
Sim_Parallel                   | 3.1614
SPOA_alignment_initial         | 2.4605
compute_mi_hybrid_execution    | 1.4547
SPOA_Consensus_Total           | 1.4402
SPOA_alignment                 | 1.1558
Real_Score                     | 0.4925
get_elbow_columns              | 0.4079
hamming_distance_matrix        | 0.0378
encode_msa                     | 0.0257
filter_consensus_generation    | 0.0242
agglomerative_clustering       | 0.0088
Fuzzy_CDIST_Calculations       | 0.0044
Filter_Invariant               | 0.0030

         ID   Barcode                                             Insert  \
629      47  CTAGGTCC  AACGCTTTAAGATGCAGGCGTCAATGAAGCTCTCCGTTGTGATGAT...   
630      47  CTAGGTCC  AACGCTTTAAGATGCAGGCGTCAATGAAGCTCTCCGGTTGTGATGA...   
631      47  CTAGGTC

/var/folders/kr/crxb3xqd4wv1ysx_0t4zgxv80000gn/T/ipykernel_10028/3654542128.py:837: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cluster_scores = result.groupby('cluster').apply(get_mean_consistency)


!!! Outlier: Real=67.1020 > P99=16.6653
Cluster 8929272 may be formed of two highly similar sequences

OPERATION                      | TIME (s)  
-------------------------------------------
MI_Integrity_Check_Total       | 5.1092
Sim_Parallel                   | 3.2677
SPOA_alignment_initial         | 2.5724
SPOA_Consensus_Total           | 1.5255
compute_mi_hybrid_execution    | 1.5166
SPOA_alignment                 | 1.2239
Real_Score                     | 0.4990
get_elbow_columns              | 0.4205
hamming_distance_matrix        | 0.0392
encode_msa                     | 0.0277
filter_consensus_generation    | 0.0252
agglomerative_clustering       | 0.0092
Fuzzy_CDIST_Calculations       | 0.0048
Filter_Invariant               | 0.0032

         ID   Barcode                                             Insert  \
735      55  TATCCCAG  TGTACACTAGTGGAGCCTCACGGCCCCAAAAGCTTCAGCACTTACC...   
736      55  TATCCCAG  TAGTACACTAGTGGAGCCTCACGGCCCCAAAAGCTTCAGCACTTAC...   
737      55  TATCCCA

/var/folders/kr/crxb3xqd4wv1ysx_0t4zgxv80000gn/T/ipykernel_10028/3654542128.py:837: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cluster_scores = result.groupby('cluster').apply(get_mean_consistency)


,ID,Barcode,Insert,cluster,mi_warning,cluster_consensus_bc,cluster_mean_barcode_sim,barcodes_consistent,cluster_consensus_insert
735,55,TATCCCAG,TGTACACTAGTGGAGCCTCACGGCCCCAAAAGCTTCAGCACTTACC...,0,True,TATCCCAG,0.995833,True,TAGTACACTAGTGGAGCCTCAGGCCCCAAAAGCTTCAGCACTTACC...
736,55,TATCCCAG,TAGTACACTAGTGGAGCCTCACGGCCCCAAAAGCTTCAGCACTTAC...,0,True,TATCCCAG,0.995833,True,TAGTACACTAGTGGAGCCTCAGGCCCCAAAAGCTTCAGCACTTACC...
737,55,TATCCCAG,TAGTACACTAGTGGAGCCTCACGGCCCCAAAAGCTTCAGCACTTAC...,0,True,TATCCCAG,0.995833,True,TAGTACACTAGTGGAGCCTCAGGCCCCAAAAGCTTCAGCACTTACC...
738,55,TATCCCAG,TAGTTCACTAGTGGAGCCTCACGGCCCCAAAAGCTTCAGCACTTAC...,0,True,TATCCCAG,0.995833,True,TAGTACACTAGTGGAGCCTCAGGCCCCAAAAGCTTCAGCACTTACC...
739,55,TATCCCAG,TAGTACACTAGTGGAGCCTCACGGCCCCAATAGCTTCAGCACTTAC...,0,True,TATCCCAG,0.995833,True,TAGTACACTAGTGGAGCCTCAGGCCCCAAAAGCTTCAGCACTTACC...
...,...,...,...,...,...,...,...,...,...
37961,3055,ATCCCAG,ATCAGCATTTGCACCGTGCATCGTGGTCTACCTGAAATCTATCAAG...,2,False,TATCCCAG,0.995833,True,ATCAGCATTTGCACCGTGCATCGTGGTCTACCTGAAATCTATCAAG...
37962,3055,TATCCCAG,ATCAGCATTTGCACCGTGCATCGTGGTCTACCTGAAATCTATCAAG...,2,False,TATCCCAG,0.995833,True,ATCAGCATTTGCACCGTGCATCGTGGTCTACCTGAAATCTATCAAG...
37963,3055,TATCCCAG,ATCAGCATTTGCACCGTGCATCGTGGTCTACCTGAAATCTATCAAG...,2,False,TATCCCAG,0.995833,True,ATCAGCATTTGCACCGTGCATCGTGGTCTACCTGAAATCTATCAAG...
37964,3055,TATCCCAG,ATCAGCATTTGCACCGTGCATCGTGGTCTACCTGAAATCTATCAAG...,2,False,TATCCCAG,0.995833,True,ATCAGCATTTGCACCGTGCATCGTGGTCTACCTGAAATCTATCAAG...


In [42]:
msa_list = ['ATCGATCG', 'ATCGATCG', 'ATCGGTCG', 'ATCGGTCG']
fast_consensus(msa_list)

'ATCGaTCG'

In [109]:
result_df2

,ID,Barcode,Insert,cluster,cluster_consensus_bc,cluster_consensus_insert
347,56,TCGCGCACAGGA,TTGGTAACGCTCTGCGACCAATAACGCGGTGCACCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
348,56,TCGCGACAGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCA,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
349,56,CCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
350,56,TCGCGAACAGGA,TTGGTACGCACTGCGAAATAACGCGGTGTCGTCAGCGTCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
351,56,TCGCGAACAGGA,TTGGTACGCTCTGTGACCAATAACGCGGTGCATCACGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
352,56,TCGCTAAACAGGA,TTGGTACGCTTGCGAGCCAATAACGCGATGCATCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
353,56,TCGCGAACAGGA,TTGGTACGCTGTGCGACAATAACGCGGTGATCAGCGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
354,56,TCGCGAACAGGA,TTGGTACGCTCTCGCGCCAATAACGCGGTGCATCAGCTGCAT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
355,56,TCGCGAACAGGA,TTGGTCCGCTCTGCGACCAATAACGCAGTGCATCAGCCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
356,56,TCGCGAACAGGA,TTGGTACTCTCTACGACCAATAACGCGGTGAATCTGTGCT,0,TCGCGAACAGGA,TTGGTACGCTCTGCGACCAATAACGCGGTGCATCAGCGCT
